# Where are the most e-bikes?

Live snapshot of Bluebikes station status: top 5 stations by e-bikes as a share of total docks, and top 5 by raw e-bike count.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from shared.api import get
from shared.plots.style import add_source_footnote, apply_theme

apply_theme()

## Params

`MIN_CAPACITY` keeps tiny stations out of the percentage ranking (1 e-bike at a 3-dock station is 33%).

In [ ]:
GBFS_URL = "https://gbfs.bluebikes.com/gbfs/gbfs.json"
FEEDS = ["station_information", "station_status"]
LANGUAGE = "en"
TOP_N = 5
MIN_CAPACITY = 10

## Pull and cache

GBFS is live, so each run writes a new snapshot keyed by UTC pull time.

In [ ]:
feed_urls = {
    feed["name"]: feed["url"]
    for feed in get(GBFS_URL)["data"][LANGUAGE]["feeds"]
}

pulled_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")

tables = {}
for feed_name in FEEDS:
    payload = get(feed_urls[feed_name])
    df = pd.json_normalize(payload["data"]["stations"])
    raw_path = Path(f"data/raw/{feed_name}/{pulled_at}/data.csv")
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(raw_path, index=False)
    print(f"Fetched and cached {len(df):,} rows: {raw_path}")
    tables[feed_name] = df

stations = tables["station_information"][["station_id", "name", "capacity"]].merge(
    tables["station_status"][["station_id", "num_ebikes_available"]],
    on="station_id",
)
stations["ebike_pct"] = stations["num_ebikes_available"] / stations["capacity"] * 100
stations = stations[stations["capacity"] > 0]
print(f"{len(stations):,} stations, {stations['num_ebikes_available'].sum():,} e-bikes available")

## Top 5 by share of total docks

In [ ]:
top_pct = (
    stations[stations["capacity"] >= MIN_CAPACITY]
    .nlargest(TOP_N, ["ebike_pct", "num_ebikes_available"])
    .reset_index(drop=True)
)
top_pct

## Top 5 by e-bike count

In [ ]:
top_count = stations.nlargest(TOP_N, ["num_ebikes_available", "ebike_pct"]).reset_index(drop=True)
top_count

## Charts

In [ ]:
out_dir = Path(f"outputs/{pulled_at}")
(out_dir / "charts").mkdir(parents=True, exist_ok=True)
(out_dir / "tables").mkdir(parents=True, exist_ok=True)
top_pct.to_csv(out_dir / "tables" / "top_by_pct.csv", index=False)
top_count.to_csv(out_dir / "tables" / "top_by_count.csv", index=False)

snapshot = datetime.strptime(pulled_at, "%Y-%m-%dT%H%M%SZ").strftime("%b %d, %Y %H:%M UTC")
SOURCE = "Bluebikes GBFS (gbfs.bluebikes.com)"


def barh(df, value, title, xlabel, fmt, filename):
    fig, ax = plt.subplots(figsize=(9, 4))
    d = df.iloc[::-1]
    bars = ax.barh(d["name"], d[value])
    ax.bar_label(bars, labels=[fmt(v) for v in d[value]], padding=3)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.margins(x=0.12)
    add_source_footnote(fig, SOURCE)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(out_dir / "charts" / filename)
    plt.show()


barh(top_pct, "ebike_pct",
     f"Top {TOP_N} Stations by E-bike Share of Docks, {snapshot}",
     f"E-bikes as % of total docks (stations with {MIN_CAPACITY}+ docks)",
     lambda v: f"{v:.0f}%", "top_by_pct.png")

barh(top_count, "num_ebikes_available",
     f"Top {TOP_N} Stations by E-bikes Available, {snapshot}",
     "E-bikes available",
     lambda v: f"{v:,.0f}", "top_by_count.png")